<a href="https://colab.research.google.com/github/N-Uma/Machine-Learning-and-Big-Data/blob/main/Data_Injestion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


print(f"PROJECT: {path}")

import os
folders = [
    "notebooks", "data/raw", "data/processed",
    "tableau", "config", "scripts", "tests"
]

for folder in folders:
    folder_path = os.path.join(path, folder)
    os.makedirs(folder_path, exist_ok=True)
    print(f"Created: {folder_path}")

print("\n FOLDER STRUCTURE:")
os.listdir(path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT: /content/drive/MyDrive/NYC_Taxi_Yellow_2012
Created: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/notebooks
Created: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/data/raw
Created: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/data/processed
Created: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/tableau
Created: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/config
Created: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/scripts
Created: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/tests

 FOLDER STRUCTURE:


['data',
 'tableau',
 'models',
 'notebooks',
 'config',
 'scripts',
 'tests',
 'final_dataset.csv']

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

path = "/content/drive/MyDrive/NYC_Taxi_Yellow_2012"


if 'spark' in locals() and spark:
    try:
        spark.stop()
        print("Existing Spark session stopped successfully.")
    except Exception as e:
        print(f"Could not stop existing Spark session (it might already be stopped or unreachable): {e}")

spark = (
    SparkSession.builder
    .appName("taxi-ingestion")
    .getOrCreate()
)

sc = spark.sparkContext
print(f"Version: {spark.version}")
print(f"Spark UI: {sc.uiWebUrl}")

Version: 4.0.2
Spark UI: http://eba9bc008242:4040


In [ ]:
raw_data = f"{path}/data/raw/"

import os
files = os.listdir(raw_data)
parquet_files = [file for file in files if file.endswith('.parquet')]
print(f"Found {len(parquet_files)} parquet files:")
for file in parquet_files[:]:
    print(f" {file}")

Found 12 parquet files:
 yellow_2012_01.parquet
 yellow_2012_02.parquet
 yellow_2012_03.parquet
 yellow_2012_04.parquet
 yellow_2012_05.parquet
 yellow_2012_06.parquet
 yellow_2012_07.parquet
 yellow_2012_08.parquet
 yellow_2012_09.parquet
 yellow_2012_10.parquet
 yellow_2012_11.parquet
 yellow_2012_12.parquet


In [ ]:
yellowtaxi_df = spark.read.parquet(f"{path}/data/raw/*.parquet")
print("\n Sample data:")
yellowtaxi_df.show(7)
print("\n Schema:")
yellowtaxi_df.printSchema()



 Sample data:
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2012-01-01 00:07:56|  2012-01-01 00:12:09|              1|          0.9|         1|                 N|         158|         231|           2|        4.9|  0.5|  

In [ ]:
from pyspark.sql.types import DoubleType, FloatType

def null_check(df):
    rows_count = df.count()
    print(f"Total rows: {rows_count:,}")

    key_cols = ["fare_amount", "trip_distance", "passenger_count",
                "tpep_pickup_datetime"]

    select_expressions = []
    for c in key_cols:
        col_type = df.schema[c].dataType
        if isinstance(col_type, (DoubleType, FloatType)):
            select_expressions.append(
                sum(when(col(c).isNull() | isnan(col(c)), 1).otherwise(0)).alias(f"{c}_nulls")
            )
        else:
            select_expressions.append(
                sum(when(col(c).isNull(), 1).otherwise(0)).alias(f"{c}_nulls")
            )

    null_summary = df.select(select_expressions).collect()[0].asDict()

    print("\n NULL PERCENTAGE:")
    for col_name, null_count in null_summary.items():
        pct = (null_count / rows_count) * 100
        print(f"  {col_name:20s}: {null_count:8,} ({pct:6.2f}%)")

    print("\n FARE AMOUNT STATS:")
    df.select("fare_amount").describe().show()

    return rows_count, null_summary

rows_count, null_summary = null_check(yellowtaxi_df)

Total rows: 171,359,007

 NULL PERCENTAGE:
  fare_amount_nulls   :        0 (  0.00%)
  trip_distance_nulls :        0 (  0.00%)
  passenger_count_nulls:        0 (  0.00%)
  tpep_pickup_datetime_nulls:        0 (  0.00%)

 FARE AMOUNT STATS:
+-------+------------------+
|summary|       fare_amount|
+-------+------------------+
|  count|         171359007|
|   mean|10.989770782162557|
| stddev| 8.923958348729043|
|    min|               2.5|
|    max|             500.0|
+-------+------------------+



In [ ]:
clean_df = yellowtaxi_df.filter(
    (col("fare_amount").between(1.0, 500.0)) &
    (col("trip_distance").between(0.1, 100.0)) &
    (col("passenger_count").cast("int").between(1, 6))
).dropDuplicates()

In [ ]:
clean_df = clean_df.withColumn("pickup_month", date_format(col("tpep_pickup_datetime"), "yyyy-MM"))

In [ ]:
import os


RAW_PATH = raw_data


if os.path.isdir(RAW_PATH):
    print(f"Contents of '{RAW_PATH}':")
    for item in os.listdir(RAW_PATH):
        print(item)
else:
    print(f"The directory '{RAW_PATH}' does not exist.")

Contents of '/content/drive/MyDrive/NYC_Taxi_Yellow_2012/data/raw/':
yellow_2012_01.parquet
yellow_2012_02.parquet
yellow_2012_03.parquet
yellow_2012_04.parquet
yellow_2012_05.parquet
yellow_2012_06.parquet
yellow_2012_07.parquet
yellow_2012_08.parquet
yellow_2012_09.parquet
yellow_2012_10.parquet
yellow_2012_11.parquet
yellow_2012_12.parquet


In [ ]:
from pyspark.sql.functions import col, date_format, year
from pyspark import StorageLevel


columns_needed = [
    "fare_amount",
    "trip_distance",
    "passenger_count",
    "tpep_pickup_datetime",
    "congestion_surcharge"
]

df_pruned = yellowtaxi_df.select(*columns_needed)


df_clean = (
    df_pruned
    .filter(col("fare_amount").between(1.0, 500.0))
    .filter(col("trip_distance").between(0.1, 100.0))
    .filter(col("passenger_count").cast("int").between(1, 6))
    .filter(year(col("tpep_pickup_datetime").cast("timestamp")) == 2012)
)


df_clean = df_clean.withColumn(
    "pickup_month",
    date_format(col("tpep_pickup_datetime"), "yyyy-MM")
)


df_clean = df_clean.persist(StorageLevel.MEMORY_AND_DISK)


print("Cleaned data ready.")

Cleaned data ready.


In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, date_format, year
from pyspark.sql.types import *
from pyspark import StorageLevel


for df_name in ['df_clean', 'df_pruned', 'yellowtaxi_df']:
    if df_name in locals() and isinstance(locals()[df_name], pyspark.sql.DataFrame):
        try:
            locals()[df_name].unpersist()
            print(f"{df_name} unpersisted.")
        except:
            pass


spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")
print("spark.sql.parquet.enableVectorizedReader set to false.")


custom_schema = StructType([
    StructField("VendorID", LongType(), True),
    StructField("tpep_pickup_datetime", TimestampType(), True),
    StructField("tpep_dropoff_datetime", TimestampType(), True),
    StructField("passenger_count", LongType(), True),
    StructField("trip_distance", DoubleType(), True),
    StructField("RatecodeID", LongType(), True),
    StructField("store_and_fwd_flag", StringType(), True),
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    StructField("payment_type", LongType(), True),
    StructField("fare_amount", DoubleType(), True),
    StructField("extra", DoubleType(), True),
    StructField("mta_tax", DoubleType(), True),
    StructField("tip_amount", DoubleType(), True),
    StructField("tolls_amount", DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("congestion_surcharge", DoubleType(), True),
    StructField("airport_fee", DoubleType(), True)
])


yellowtaxi_df = spark.read.schema(custom_schema).parquet(f"{path}/data/raw/*.parquet")
print("yellowtaxi_df re-read successfully with custom schema.")


columns_needed = [
    "fare_amount",
    "trip_distance",
    "passenger_count",
    "tpep_pickup_datetime",
    "congestion_surcharge"
]

df_pruned = yellowtaxi_df.select(*columns_needed)



df_clean = (
    df_pruned
    .filter(col("fare_amount").between(1.0, 500.0))
    .filter(col("trip_distance").between(0.1, 100.0))
    .filter(col("passenger_count").cast("int").between(1, 6))
    .filter(year(col("tpep_pickup_datetime").cast("timestamp")) == 2012)
)


print("df_clean DataFrame created (lazily).")

print("\nSchema of df_clean:")
df_clean.printSchema()

print("\nSample data from df_clean:")
df_clean.show(7)

df_clean unpersisted.
df_pruned unpersisted.
yellowtaxi_df unpersisted.
spark.sql.parquet.enableVectorizedReader set to false.
yellowtaxi_df re-read successfully with custom schema.
df_clean DataFrame created (lazily).

Schema of df_clean:
root
 |-- fare_amount: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- congestion_surcharge: double (nullable = true)


Sample data from df_clean:
+-----------+-------------+---------------+--------------------+--------------------+
|fare_amount|trip_distance|passenger_count|tpep_pickup_datetime|congestion_surcharge|
+-----------+-------------+---------------+--------------------+--------------------+
|        4.9|          0.9|              1| 2012-01-01 00:07:56|                NULL|
|        8.5|          2.3|              1| 2012-01-01 00:18:49|                NULL|
|        9.3|          2.2|              1| 2012-01-01 0

In [ ]:
from pyspark.sql.functions import col, date_format


PROCESSED_PATH = f"{path}/data/processed/"


spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")
print("spark.sql.parquet.enableVectorizedReader set to false before writing.")


df_clean_with_month = df_clean.withColumn(
    "pickup_month",
    date_format(col("tpep_pickup_datetime"), "yyyy-MM")
)


(
    df_clean_with_month
    .repartition(50)
    .write
    .mode("overwrite")
    .option("compression", "snappy")
    .partitionBy("pickup_month")
    .parquet(PROCESSED_PATH)
)

print(f" Data successfully written to: {PROCESSED_PATH}")

spark.sql.parquet.enableVectorizedReader set to false before writing.
 Data successfully written to: /content/drive/MyDrive/NYC_Taxi_Yellow_2012/data/processed/


In [ ]:

df_final = spark.read.parquet(PROCESSED_PATH)

print("--- Partition Distribution (Expected: only 2012 data) ---")

(
    df_final
    .groupBy("pickup_month")
    .count()
    .orderBy("pickup_month")
    .show(12, truncate=False)
)

--- Partition Distribution (Expected: only 2012 data) ---
+------------+--------+
|pickup_month|count   |
+------------+--------+
|2012-01     |12689698|
|2012-02     |12979847|
|2012-03     |15685434|
|2012-04     |12995743|
|2012-05     |13846759|
|2012-06     |14966148|
|2012-07     |14257696|
|2012-08     |14257648|
|2012-09     |14430991|
|2012-10     |14407354|
|2012-11     |13663801|
|2012-12     |14572953|
+------------+--------+

